# Transformer — The Library Version

Three parts, honestly framed: (1) our causal multi-head attention verified against a naive per-position, per-head loop; (2) the baseline contrast that locates the achievement precisely; (3) the PyTorch translation — shown, not run.

In [1]:
import numpy as np
import pandas as pd

def softmax(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=-1, keepdims=True)

# (1) vectorized causal MHA vs the slow honest loop
rs = np.random.default_rng(0)
m, T, D, H = 4, 10, 48, 4
DH = D // H
X = rs.normal(0, 1, (m, T, D))
Wq, Wk, Wv = (rs.normal(0, 0.15, (D, D)) for _ in range(3))
MASK = np.triu(np.ones((T, T)), k=1) * -1e9

def split(x): return x.reshape(m, T, H, DH).transpose(0, 2, 1, 3)
Q, K, V = split(X @ Wq), split(X @ Wk), split(X @ Wv)
O_vec = softmax(Q @ K.transpose(0, 1, 3, 2) / np.sqrt(DH) + MASK) @ V

O_loop = np.zeros_like(O_vec)
for i in range(m):
    for h in range(H):
        for t in range(T):
            scores = np.array([Q[i, h, t] @ K[i, h, u] / np.sqrt(DH) if u <= t else -1e9 for u in range(T)])
            w = np.exp(scores - scores.max()); w /= w.sum()
            O_loop[i, h, t] = sum(w[u] * V[i, h, u] for u in range(T))
print(f"vectorized vs naive triple-loop — max |difference|: {np.abs(O_vec - O_loop).max():.2e}")
print("(identical: heads are independent attention problems; the mask is just -inf before softmax)")

vectorized vs naive triple-loop — max |difference|: 4.44e-16
(identical: heads are independent attention problems; the mask is just -inf before softmax)


In [2]:
# (2) the honest baseline: WHERE exactly is the achievement?
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

tr = pd.read_csv("data/addition_train.csv"); te = pd.read_csv("data/addition_test.csv")
def feats(df):
    a = df.text.str[:2].astype(int); b = df.text.str[3:5].astype(int)
    X = np.stack([a//10, a%10, b//10, b%10], 1)
    return np.eye(10)[X].reshape(len(df), -1), (a+b).values
Xtr_f, ytr_f = feats(tr); Xte_f, yte_f = feats(te)
mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=400, random_state=0).fit(Xtr_f, ytr_f)
acc = (mlp.predict(Xte_f) == yte_f).mean()
print(f"sklearn MLP given PARSED digits (structured in -> class out): {acc:.1%} exact match")
print()
print("So addition itself isn't hard — given parsed inputs, lesson 10's machinery learns it fine.")
print("The transformer's achievement is different in kind: it received RAW CHARACTERS with no")
print("notion of number, operator, or answer position; discovered the format, the parsing, and")
print("the arithmetic from next-token loss alone; and can WRITE its answers as text. One model,")
print("one objective, zero task-specific engineering — that property is what scales to everything.")

sklearn MLP given PARSED digits (structured in -> class out): 97.7% exact match

So addition itself isn't hard — given parsed inputs, lesson 10's machinery learns it fine.
The transformer's achievement is different in kind: it received RAW CHARACTERS with no
notion of number, operator, or answer position; discovered the format, the parsing, and
the arithmetic from next-token loss alone; and can WRITE its answers as text. One model,
one objective, zero task-specific engineering — that property is what scales to everything.


### (3) The PyTorch translation — read it; you have built every line

```python
import torch, torch.nn as nn

block = nn.TransformerEncoderLayer(
    d_model=48, nhead=4, dim_feedforward=128,
    batch_first=True, norm_first=True)          # norm_first=True = our pre-LN (README §3.1)
model = nn.TransformerEncoder(block, num_layers=2)
causal = nn.Transformer.generate_square_subsequent_mask(10)   # Block 4's triangle
out = model(x, mask=causal)                      # Blocks 5-7, autograd included

# The real-world family portrait, all this exact skeleton:
#   GPT-2/3/4, Llama, Claude ... : decoder-only, causal, next-token — THIS notebook, scaled
#   differences at scale: rotary/relative positions (lesson 13 §6's fix), RMSNorm (LayerNorm
#   minus the mean-centering), SwiGLU FFNs (fancier bends), and 10^6x the parameters
```

Everything you trained today — mask, pre-LN, residuals, heads, the objective — is the production recipe. The rest is scale and refinements you now have the vocabulary to read.